# Task 3 · Augmentation
Random crop (shift of up to 4 px), rotation (up to 15°), zoom (up to 1.4×) and horizontal and vertical flips, drawn
anew for every image in every epoch, so each epoch sees 40,000 new images. The Task 1 CNN is retrained for 15 epochs
without augmentation, with all five transformations, and with all but the vertical flip.

In [ ]:
from hw3 import *

B = json.load(open(os.path.join(OUT, 'best.json')))['cnn']
X, Y, Xte, Yte = cifar()
torch.manual_seed(SEED)
ops = ['original', 'crop', 'rotation', 'zoom', 'h-flip', 'v-flip', 'all five']
fig, axes = plt.subplots(4, len(ops), figsize=(7.2, 4.3))
for r, c in enumerate((0, 3, 7, 9)):
    i = int((Y == c).nonzero()[0])
    for ax, op in zip(axes[r], ops):
        img = augment(X[i:i + 1], AUG['all'] if op == 'all five' else (op,), p=.5 if op == 'all five' else 1)[0]
        ax.imshow((img / 255).clamp(0, 1).permute(1, 2, 0))
        ax.set_title(op if r == 0 else '')
        ax.axis('off')
save(fig, 'transforms')

In [ ]:
E = 15
base = dict(model=partial(CNN, B['arch']), lr=B['lr'], epochs=E)
R = cv({'aug_none': base, 'aug_novflip': base | dict(aug='novflip'), 'aug_all': base | dict(aug='all')})
G = {'no augmentation': ('aug_none', COL['cnn']), 'all but v-flip': ('aug_novflip', COL['novflip']),
     'all five': ('aug_all', COL['all'])}
pd.DataFrame({k: {m: ms(col(R[n], m)) for m in ('train_acc', 'val_acc', 'test_acc')} for k, (n, _) in G.items()}).T

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.3), gridspec_kw=dict(width_ratios=[1, 1, .7]))
curves({k: (R[n], c) for k, (n, c) in G.items()}, axes[:2])
for x, (n, c) in enumerate(G.values()):
    a = col(R[n], 'test_acc') * 100
    axes[2].errorbar(x, a.mean(), a.std(ddof=1), fmt='o', capsize=4, color=c)
axes[2].set(xticks=range(3), xlim=(-.5, 2.5), ylabel='test accuracy (%)')
axes[2].set_xticklabels(['none', 'no v-flip', 'all five'], rotation=20)
save(fig, 'aug_curves')

In [ ]:
conf = lambda n: np.mean([np.bincount(10 * Yte.numpy() + r['pred'].numpy(), minlength=100).reshape(10, 10) / 10
                          for r in R[n]], 0)
name = {n: k for k, (n, _) in G.items()}
top = max(['aug_novflip', 'aug_all'], key=lambda n: col(R[n], 'test_acc').mean())
D = conf(top) - conf('aug_none')
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.8))
for ax, M, t, cmap, lim in zip(axes, (conf('aug_none'), conf(top), D), ('no augmentation (%)', f'{name[top]} (%)', 'difference (pp)'),
                               ('Blues', 'Blues', 'RdBu'), (100, 100, np.abs(D).max())):
    ax.imshow(M, cmap=cmap, vmin=-lim if cmap == 'RdBu' else 0, vmax=lim)
    for (r, c), v in np.ndenumerate(M):
        ax.text(c, r, f'{v:.0f}', ha='center', va='center', fontsize=4.5, color='white' if abs(v) > .6 * lim else INK)
    ax.set_xticks(range(10), CLASSES, rotation=90, fontsize=6)
    ax.set_yticks(range(10), CLASSES if ax is axes[0] else [], fontsize=6)
    ax.set_title(t)
    ax.grid(False)
save(fig, 'confusion')

In [ ]:
pc = {n: np.array([[(r['pred'] == Yte)[Yte == c].float().mean().item() for c in range(10)] for r in R[n]]) * 100
      for n, _ in G.values()}
fig, ax = plt.subplots(figsize=(7.2, 2))
for j, (k, (n, c)) in enumerate(G.items()):
    ax.errorbar(np.arange(10) + .22 * (j - 1), pc[n].mean(0), pc[n].std(0, ddof=1), fmt='o', ms=3.5, capsize=2, color=c, label=k)
ax.set_xticks(range(10), CLASSES)
ax.set(ylabel='test accuracy (%)')
ax.legend(ncol=3)
save(fig, 'per_class')

In [ ]:
gain = pc[top].mean(0) - pc['aug_none'].mean(0)
tex('t3', epochs=E, topName=name[top], bestClass=CLASSES[gain.argmax()], bestGain=f'{gain.max():+.1f}',
    worstClass=CLASSES[gain.argmin()], worstGain=f'{gain.min():+.1f}',
    **{m + n[4:].capitalize(): ms(f(R[n])) for n in R for m, f in (
        ('test', lambda x: col(x, 'test_acc')), ('val', lambda x: col(x, 'val_acc')),
        ('gap', lambda x: col(x, 'train_acc') - col(x, 'val_acc')))})